# **3일차 팀 프로젝트: 테이블 데이터 조회 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 CSV 테이블 데이터를 Supabase에 적재
2. SQL 쿼리로 데이터 조회 테스트
3. Text2SQL 시스템 구현 및 테스트

## 구현 단계
- 환경 설정 확인
- CSV 파일 확인 및 탐색
- Supabase 연결
- CSV 데이터 업로드
- SQL 쿼리 테스트
- Text2SQL 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

# OpenAI API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Supabase 설정 확인
if os.environ.get("SUPABASE_DB_URL"):
    print("✓ Supabase DB URL이 설정되었습니다.")
else:
    print("✗ Supabase DB URL이 필요합니다.")
    print("  .env 파일에 SUPABASE_DB_URL을 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Supabase DB URL이 설정되었습니다.


## 1. CSV 파일 확인 및 탐색

**TODO: 팀에서 준비한 CSV 파일 경로를 입력하세요**

**중요:** 여러 CSV 파일을 사용하는 경우, 테이블 간 관계(Foreign Key)를 고려하여 업로드 순서를 결정하세요.
- 부모 테이블 → 자식 테이블 순서로 업로드

In [3]:
import pandas as pd

# TODO: 팀의 CSV 파일 경로를 입력하세요
# 여러 파일이 있다면 dict 형태로 구성
# 예시:
# csv_files = {
#     "table1": "../datasets/your_table1.csv",
#     "table2": "../datasets/your_table2.csv"
# }

csv_files = {
    "cve": r"C:\Users\SAMSUNG\OneDrive\문서\카카오톡 받은 파일\cve.csv",
    "cwe": r"C:\Users\SAMSUNG\OneDrive\문서\카카오톡 받은 파일\cwe.csv"
    # 필요한 만큼 추가
}

# CSV 파일 로드 및 확인
dataframes = {}

for table_name, file_path in csv_files.items():
    try:
        df = pd.read_csv(file_path)
        dataframes[table_name] = df

        print("=" * 80)
        print(f"📋 {table_name} 테이블")
        print("=" * 80)
        print(f"\n행 수: {len(df)}")
        print(f"컬럼: {list(df.columns)}")
        print(f"\n첫 5개 행:")
        print(df.head())
        print(f"\n데이터 타입:")
        print(df.dtypes)
        print("\n")

    except Exception as e:
        print(f"✗ {table_name} 로드 실패: {e}\n")

print(f"\n✓ 총 {len(dataframes)}개의 테이블 로드 완료")

📋 cve 테이블

행 수: 1671
컬럼: ['cveID', 'vendorProject', 'product', 'vulnerabilityName', 'dateAdded', 'shortDescription', 'requiredAction', 'dueDate', 'knownRansomwareCampaignUse', 'notes', 'cwes']

첫 5개 행:
            cveID vendorProject  \
0  CVE-2026-64849        MLflow   
1  CVE-2026-33824     Microsoft   
2  CVE-2026-59310      Broadcom   
3  CVE-2026-55040     Microsoft   
4  CVE-2026-65400         Apple   

                                          product  \
0                                          MLflow   
1  Internet Key Exchange (IKE) Service Extensions   
2                                  VMware vCenter   
3                                      SharePoint   
4                                           macOS   

                                   vulnerabilityName   dateAdded  \
0   MLflow Server-Side Request Forgery Vulnerability  2026-08-19   
1  Microsoft Internet Key Exchange (IKE) Service ...  2026-08-18   
2  Broadcom VMware vCenter Path Traversal Vulnera...  2026-08-18

## 2. 데이터 탐색 및 통계

**TODO: 팀 데이터에 맞는 탐색 쿼리를 작성하세요**

In [4]:
# TODO: 각 테이블의 주요 통계를 확인하세요
# 예시:
# - 특정 컬럼의 고유값 개수
# - 카테고리별 데이터 분포
# - 결측치 확인

for table_name, df in dataframes.items():
    print(f"\n{'='*80}")
    print(f"📊 {table_name} 통계")
    print("="*80)

    # 기본 통계
    print("\n[기본 정보]")
    print(df.info())

    # 결측치 확인
    print("\n[결측치]")
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print(null_counts[null_counts > 0])
    else:
        print("결측치 없음")

    # TODO: 팀 데이터에 맞는 추가 탐색 코드를 작성하세요
    # 예시:
    # print("\n[카테고리 분포]")
    # print(df['YOUR_COLUMN'].value_counts())


📊 cve 통계

[기본 정보]
<class 'pandas.DataFrame'>
RangeIndex: 1671 entries, 0 to 1670
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   cveID                       1671 non-null   str  
 1   vendorProject               1671 non-null   str  
 2   product                     1671 non-null   str  
 3   vulnerabilityName           1671 non-null   str  
 4   dateAdded                   1671 non-null   str  
 5   shortDescription            1671 non-null   str  
 6   requiredAction              1671 non-null   str  
 7   dueDate                     1671 non-null   str  
 8   knownRansomwareCampaignUse  1671 non-null   str  
 9   notes                       1671 non-null   str  
 10  cwes                        1500 non-null   str  
dtypes: str(11)
memory usage: 1.0 MB
None

[결측치]
cwes    171
dtype: int64

📊 cwe 통계

[기본 정보]
<class 'pandas.DataFrame'>
Index: 969 entries, 5 to 1434
Data columns (total 

## 3. Supabase PostgreSQL 연결

In [5]:
from langchain_community.utilities import SQLDatabase

supabase_db_url = os.getenv("SUPABASE_DB_URL")

if not supabase_db_url:
    raise Exception("SUPABASE_DB_URL이 설정되지 않았습니다. .env 파일을 확인하세요.")

print("Supabase PostgreSQL 연결 중...\n")

try:
    # LangChain SQLDatabase로 PostgreSQL 연결
    db = SQLDatabase.from_uri(supabase_db_url)

    print("✓ PostgreSQL 연결 성공!\n")
    print("현재 테이블 목록:")
    tables = db.get_usable_table_names()
    print(tables)

except Exception as e:
    print(f"✗ 연결 실패: {e}")
    raise

C:\Users\SAMSUNG\AppData\Local\Temp\ipykernel_20372\1442992373.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Supabase PostgreSQL 연결 중...

✓ PostgreSQL 연결 성공!

현재 테이블 목록:
['CVE', 'CWE', 'brain']


## 4. Supabase에 CSV 데이터 업로드

### Supabase 대시보드 (GUI)
1. Supabase 대시보드 접속
2. Table Editor → Import data from CSV
3. **반드시 테이블 관계 순서대로 업로드** (부모 → 자식)
4. Foreign Key 에러 발생 시 순서를 재확인

## 5. 업로드 확인 및 스키마 탐색

In [16]:
# 데이터베이스 다시 연결 (업로드 후 스키마 갱신)
db = SQLDatabase.from_uri(supabase_db_url)

print("=== 데이터베이스 스키마 ===")
print(db.table_info)

print("\n" + "="*80 + "\n")

# 각 테이블의 샘플 데이터
for table in db.get_usable_table_names():
    print(f"{table} 테이블 샘플:")
    try:
        result = db.run(f"SELECT * FROM {table} LIMIT 3")
        print(result)
    except Exception as e:
        print(f"조회 실패: {e}")
    print()

=== 데이터베이스 스키마 ===

CREATE TABLE "CVE" (
	"cveID" TEXT, 
	"vendorProject" TEXT, 
	product TEXT, 
	"vulnerabilityName" TEXT, 
	"dateAdded" TEXT, 
	"shortDescription" TEXT, 
	"requiredAction" TEXT, 
	"dueDate" TEXT, 
	"knownRansomwareCampaignUse" TEXT, 
	notes TEXT, 
	cwes TEXT
)

/*
3 rows from CVE table:
cveID	vendorProject	product	vulnerabilityName	dateAdded	shortDescription	requiredAction	dueDate	knownRansomwareCampaignUse	notes	cwes
CVE-2026-64849	MLflow	MLflow	MLflow Server-Side Request Forgery Vulnerability	2026-08-19	MLflow contains a server-side request forgery vulnerability that can allow attackers to reach intern	Apply mitigations in accordance with vendor instructions, ensuring compliance with CISA’s BOD 26-04 	2026-09-02	Unknown	https://github.com/mlflow/mlflow/pull/24258 ; https://github.com/mlflow/mlflow/issues/24179 ; BOD 26	CWE-918
CVE-2026-33824	Microsoft	Internet Key Exchange (IKE) Service Extensions	Microsoft Internet Key Exchange (IKE) Service Extensions Double Free 

## 6. SQL 쿼리 테스트

**TODO: 팀 데이터에 맞는 SQL 쿼리를 작성하여 테스트하세요**

In [18]:
# TODO: 기본 조회 쿼리 작성
# 예시: 특정 조건으로 데이터 조회

query = """
SELECT "cveID", "vulnerabilityName"
FROM "CVE"
WHERE "cveID" IS NOT NULL
LIMIT 10;
"""

print("실행 쿼리:")
print(query)
print("\n결과:")

try:
    result = db.run(query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT "cveID", "vulnerabilityName"
FROM "CVE"
WHERE "cveID" IS NOT NULL
LIMIT 10;


결과:
[('CVE-2026-64849', 'MLflow Server-Side Request Forgery Vulnerability'), ('CVE-2026-33824', 'Microsoft Internet Key Exchange (IKE) Service Extensions Double Free Vulnerability'), ('CVE-2026-59310', 'Broadcom VMware vCenter Path Traversal Vulnerability'), ('CVE-2026-55040', 'Microsoft SharePoint Weak Authentication Vulnerability'), ('CVE-2026-65400', 'Apple macOS Improper Authentication Vulnerability'), ('CVE-2025-62593', 'Ray-Project Ray Code Injection Vulnerability'), ('CVE-2025-6218', 'RARLAB WinRAR Path Traversal Vulnerability'), ('CVE-2026-20349', 'Cisco Secure Firewall Adaptive Security Appliance (ASA) and Secure Firewall Threat Defense (FTD) Heap Inspection Vulnerability'), ('CVE-2026-68820', 'Microsoft Windows Ancillary Function Driver for WinSock Use-After-Free Vulnerability'), ('CVE-2026-72898', 'Metabase SQL Injection Vulnerability')]


In [19]:
# TODO: JOIN 쿼리 작성 (여러 테이블을 사용하는 경우)
# 예시: 두 테이블을 조인하여 데이터 조회

join_query = """
SELECT "vendorProject", COUNT(*) AS count
FROM "CVE"
GROUP BY "vendorProject"
ORDER BY count DESC
LIMIT 10;
"""

print("실행 쿼리:")
print(join_query)
print("\n결과:")

try:
    result = db.run(join_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT "vendorProject", COUNT(*) AS count
FROM "CVE"
GROUP BY "vendorProject"
ORDER BY count DESC
LIMIT 10;


결과:
[('Microsoft', 385), ('Cisco', 96), ('Apple', 94), ('Adobe', 80), ('Google', 72), ('Oracle', 45), ('Apache', 40), ('Ivanti', 35), ('Fortinet', 29), ('Linux', 26)]


In [20]:
# TODO: 집계(Aggregation) 쿼리 작성
# 예시: GROUP BY, COUNT, SUM, AVG 등 사용

aggregation_query = """
SELECT "vendorProject", COUNT(*) AS count
FROM "CVE"
GROUP BY "vendorProject"
ORDER BY count DESC
LIMIT 10;
"""

print("실행 쿼리:")
print(aggregation_query)
print("\n결과:")

try:
    result = db.run(aggregation_query)
    print(result)
except Exception as e:
    print(f"쿼리 실행 오류: {e}")

실행 쿼리:

SELECT "vendorProject", COUNT(*) AS count
FROM "CVE"
GROUP BY "vendorProject"
ORDER BY count DESC
LIMIT 10;


결과:
[('Microsoft', 385), ('Cisco', 96), ('Apple', 94), ('Adobe', 80), ('Google', 72), ('Oracle', 45), ('Apache', 40), ('Ivanti', 35), ('Fortinet', 29), ('Linux', 26)]


## 7. Text2SQL 함수 구현

**TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요**

In [23]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage

llm = init_chat_model("gpt-5.4-mini")

def text_to_sql(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문을 SQL로 변환
    """
    # TODO: 시스템 프롬프트를 팀 데이터에 맞게 수정하세요
    system_prompt = f"""
    - 반드시 데이터베이스 스키마에 존재하는 테이블명과 컬럼명만 사용
    - 모든 테이블명과 컬럼명은 반드시 큰따옴표(")로 감싸기
    - CVE 테이블은 반드시 "CVE"라고 작성
    - CWE 테이블은 반드시 "CWE"라고 작성
    당신은 SQL 전문가입니다.
    사용자의 질문을 SQL 쿼리로 변환하세요.

    데이터베이스 스키마:
    {db.table_info}

    규칙:
    - PostgreSQL 문법 사용
    - SELECT 쿼리만 생성 (INSERT, UPDATE, DELETE 금지)
    - SQL 코드만 반환 (설명 불필요)
    - 코드 블록(```) 없이 순수 SQL만 반환
    - 세미콜론(;)으로 끝내기

    사용 가능한 SQL 문법:
    - JOIN (INNER, LEFT, RIGHT, FULL)
    - GROUP BY, HAVING
    - 집계 함수 (COUNT, SUM, AVG, MIN, MAX)
    - 서브쿼리
    - WHERE, ORDER BY, LIMIT
    - CTE (WITH 절)
    - 윈도우 함수
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=question)
    ]

    response = llm.invoke(messages)
    sql = response.content.strip()

    # 코드 블록 제거
    if sql.startswith("```"):
        lines = sql.split("\n")
        sql = "\n".join(lines[1:-1]) if len(lines) > 2 else sql
        sql = sql.replace("sql", "").replace("```", "").strip()

    return sql

print("✓ Text2SQL 함수 준비 완료")

✓ Text2SQL 함수 준비 완료


## 8. Text2SQL 테스트

**TODO: 팀 데이터에 맞는 자연어 질문으로 테스트하세요**

In [24]:
# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "sqli 취약점에 대해 알려줘"

print(f"질문: {question}\n")
print("="*80)

# SQL 생성
sql = text_to_sql(question, db)
print(f"\n생성된 SQL:")
print(sql)
print()

# SQL 실행
print("="*80)
print("\n실행 결과:")
try:
    result = db.run(sql)
    print(result)
except Exception as e:
    print(f"실행 오류: {e}")

질문: sqli 취약점에 대해 알려줘


생성된 SQL:
SELECT "cveID", "vendorProject", "product", "vulnerabilityName", "dateAdded", "shortDescription", "requiredAction", "dueDate", "knownRansomwareCampaignUse", "notes", "cwes"
FROM "CVE"
WHERE "vulnerabilityName" ILIKE '%SQL injection%'
   OR "shortDescription" ILIKE '%SQL injection%'
   OR "notes" ILIKE '%SQL injection%'
   OR "cwes" LIKE '%CWE-89%'
ORDER BY "dateAdded" DESC;


실행 결과:
[('CVE-2026-72898', 'Metabase', 'Metabase', 'Metabase SQL Injection Vulnerability', '2026-08-11', 'Metabase contains a SQL Injection vulnerability that allows an unauthenticated remote attacker to inject arbitrary SQL into the Metabase application database, which can give them administrator access to the instance. From there, the attacker could change the application configuration, steal...', 'Apply mitigations in accordance with vendor instructions, ensuring compliance with CISA’s BOD 26-04 Prioritizing Security Updates Based on Risk (see URL in Notes) guidance and CISA’s “F

## 9. 완전한 Text2SQL 시스템 (SQL 실행 + 자연어 답변)

**TODO: 답변 생성 프롬프트를 팀 데이터에 맞게 수정하세요**

In [25]:
def query_database(question: str, db: SQLDatabase) -> str:
    """
    자연어 질문 → SQL 생성 → 실행 → 자연어 답변
    """
    # 1. SQL 생성
    print(f"[1] SQL 생성 중...")
    sql = text_to_sql(question, db)
    print(f"    {sql}\n")

    # 2. SQL 실행
    print(f"[2] SQL 실행 중...")
    try:
        result = db.run(sql)
        print(f"    실행 완료\n")
    except Exception as e:
        return f"SQL 실행 오류: {e}"

    # 3. 자연어 답변 생성
    print(f"[3] 답변 생성 중...")

    # TODO: 시스템 프롬프트를 팀 데이터 도메인에 맞게 수정하세요
    system_prompt = """
    당신은 [YOUR_DOMAIN] 데이터 분석 전문가입니다.
    SQL 쿼리 결과를 기반으로 사용자의 질문에 자연스럽게 답변하세요.
    답변은 명확하고 이해하기 쉽게 작성하세요.
    """

    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"""
        질문: {question}

        실행한 SQL:
        {sql}

        쿼리 결과:
        {result}

        위 결과를 바탕으로 질문에 답변해주세요.
        """)
    ]

    response = llm.invoke(messages)
    return response.content

print("✓ 완전한 Text2SQL 시스템 준비 완료")

✓ 완전한 Text2SQL 시스템 준비 완료


In [26]:
from IPython.display import Markdown, display

# TODO: 팀 데이터에 맞는 질문을 작성하세요
question = "YOUR_QUESTION_HERE"

print(f"질문: {question}\n")
print("="*80 + "\n")

answer = query_database(question, db)

print("\n" + "="*80)
print("\n답변:")
display(Markdown(answer))

질문: YOUR_QUESTION_HERE


[1] SQL 생성 중...
    SELECT * FROM "CVE" LIMIT 1;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...


답변:


조회된 CVE 1건의 주요 내용은 다음과 같습니다.

- **CVE ID:** CVE-2026-64849
- **대상 제품:** MLflow
- **취약점 제목:** MLflow Server-Side Request Forgery Vulnerability
- **등록일:** 2026-08-19
- **설명:** MLflow에 **서버 측 요청 위조(SSRF)** 취약점이 있어, 공격자가 **내부망 또는 클라우드 메타데이터 서비스**에 접근하고 `response_status`, `response_body`를 받을 수 있습니다.
- **대응 권고:** 벤더 지침에 따라 완화 조치를 적용하고, CISA의 **BOD 26-04** 및 **Forensics Triage Requirements** 지침을 준수하라고 안내하고 있습니다.
- **공개일로 보이는 날짜:** 2026-09-02
- **심각도/위험도:** `Unknown`
- **CWE:** **CWE-918** (SSRF)

원하시면 이 CVE를 기준으로 **위험도 해석**, **영향 범위**, 또는 **대응 우선순위**까지 정리해드릴 수 있습니다.

## 10. 다양한 질문으로 테스트

**TODO: 최소 5개 이상의 다양한 질문으로 테스트하세요**

In [28]:
# TODO: 팀 데이터에 맞는 다양한 질문들을 작성하세요
# 기본 조회, JOIN, 집계, 정렬 등 다양한 유형의 질문 포함

questions = [
    "실제로 악용된 취약점이 가장 많은 제조사 10곳을 알려줘",
    "랜섬웨어 캠페인에 사용된 것으로 확인된 최근 취약점을 알려줘",
    "실제 악용 취약점과 가장 많이 연결된 CWE 유형 10개는?",
    "2026년에 등록된 취약점을 월별로 집계해줘",
    "CWE-89에 해당하는 실제 악용 취약점을 최근 순으로 보여줘"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print("="*80 + "\n")

    try:
        answer = query_database(q, db)
        print("\n답변:")
        display(Markdown(answer))
    except Exception as e:
        print(f"오류: {e}")


질문: 실제로 악용된 취약점이 가장 많은 제조사 10곳을 알려줘

[1] SQL 생성 중...
    SELECT "vendorProject", COUNT(*) AS "actual_exploited_count"
FROM "CVE"
WHERE "knownRansomwareCampaignUse" = 'Known'
GROUP BY "vendorProject"
ORDER BY "actual_exploited_count" DESC, "vendorProject" ASC
LIMIT 10;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


실제로 악용된 취약점이 가장 많은 제조사 10곳은 다음과 같습니다.

1. **Microsoft**: 112건  
2. **Fortinet**: 14건  
3. **Oracle**: 13건  
4. **Ivanti**: 12건  
5. **SonicWall**: 12건  
6. **Adobe**: 10건  
7. **QNAP**: 9건  
8. **VMware**: 9건  
9. **Apache**: 8건  
10. **Atlassian**: 8건  

가장 많은 제조사는 **Microsoft**로, 다른 제조사들보다 압도적으로 많은 실제 악용 사례가 확인되었습니다.


질문: 랜섬웨어 캠페인에 사용된 것으로 확인된 최근 취약점을 알려줘

[1] SQL 생성 중...
    SELECT "cveID", "vendorProject", "product", "vulnerabilityName", "dateAdded", "dueDate", "knownRansomwareCampaignUse"
FROM "CVE"
WHERE "knownRansomwareCampaignUse" = 'Known'
ORDER BY "dateAdded" DESC;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


최근 랜섬웨어 캠페인에 사용된 것으로 확인된 취약점 중 **가장 최근에 추가된 것들**은 아래와 같습니다.

### 최근 취약점
1. **CVE-2026-15409**  
   - **제품:** SonicWall SMA1000 Appliances  
   - **취약점명:** Server-Side Request Forgery(SSRF) 취약점  
   - **추가일:** 2026-07-14

2. **CVE-2026-15410**  
   - **제품:** SonicWall SMA1000 Appliances  
   - **취약점명:** Code Injection 취약점  
   - **추가일:** 2026-07-14

3. **CVE-2026-45659**  
   - **제품:** Microsoft SharePoint Server  
   - **취약점명:** Untrusted Data 역직렬화 취약점  
   - **추가일:** 2026-07-01

4. **CVE-2026-12569**  
   - **제품:** PTC Windchill and FlexPLM  
   - **취약점명:** Improper Input Validation 취약점  
   - **추가일:** 2026-06-25

5. **CVE-2026-35273**  
   - **제품:** Oracle PeopleSoft Enterprise PeopleTools  
   - **취약점명:** Critical Function에 대한 인증 누락 취약점  
   - **추가일:** 2026-06-12

### 요약
쿼리 결과 기준으로, **가장 최근 랜섬웨어 캠페인 연관 취약점은 SonicWall SMA1000 Appliances 관련 CVE-2026-15409, CVE-2026-15410**입니다.  
그 다음으로는 **Microsoft SharePoint Server, PTC Windchill/FlexPLM, Oracle PeopleSoft** 관련 취약점이 최근에 등록되었습니다.

원하시면 제가 이 목록을 **제품별로 묶어서** 또는 **위험도가 높아 보이는 순서로** 다시 정리해드릴게요.


질문: 실제 악용 취약점과 가장 많이 연결된 CWE 유형 10개는?

[1] SQL 생성 중...
    SELECT "cwes" AS "CWE", COUNT(*) AS "CVE_Count"
FROM "CVE"
WHERE "cwes" IS NOT NULL
GROUP BY "cwes"
ORDER BY COUNT(*) DESC
LIMIT 10;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


실제 악용 취약점과 가장 많이 연결된 CWE 유형 10개는 다음과 같습니다.

1. **CWE-78** — OS Command Injection: **93건**
2. **CWE-20** — Improper Input Validation: **91건**
3. **CWE-416** — Use After Free: **90건**
4. **CWE-119** — Improper Restriction of Operations within the Bounds of a Memory Buffer: **84건**
5. **CWE-787** — Out-of-bounds Write: **84건**
6. **CWE-22** — Path Traversal: **71건**
7. **CWE-94** — Code Injection: **62건**
8. **CWE-502** — Deserialization of Untrusted Data: **61건**
9. **CWE-287** — Improper Authentication: **37건**
10. **CWE-284** — Improper Access Control: **33건**

### 요약
가장 많이 등장한 유형은 **명령어 주입(CWE-78)** 이고, 그 뒤를 **입력 검증 실패(CWE-20)**, **Use After Free(CWE-416)**, **메모리 경계 관련 취약점(CWE-119, CWE-787)** 이 따르고 있습니다.  
즉, 실제 악용 사례에서는 **입력 처리 문제, 메모리 안전성 문제, 인증/권한 통제 문제**가 매우 큰 비중을 차지합니다.


질문: 2026년에 등록된 취약점을 월별로 집계해줘

[1] SQL 생성 중...
    SELECT EXTRACT(MONTH FROM TO_DATE("dateAdded", 'YYYY-MM-DD')) AS "month", COUNT(*) AS "count"
FROM "CVE"
WHERE EXTRACT(YEAR FROM TO_DATE("dateAdded", 'YYYY-MM-DD')) = 2026
GROUP BY EXTRACT(MONTH FROM TO_DATE("dateAdded", 'YYYY-MM-DD'))
ORDER BY "month";

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


2026년에 등록된 취약점을 월별로 집계하면 다음과 같습니다.

- 1월: 17건
- 2월: 28건
- 3월: 26건
- 4월: 31건
- 5월: 21건
- 6월: 23건
- 7월: 26건
- 8월: 15건

가장 많이 등록된 달은 **4월(31건)**이고, 가장 적은 달은 **8월(15건)**입니다.


질문: CWE-89에 해당하는 실제 악용 취약점을 최근 순으로 보여줘

[1] SQL 생성 중...
    SELECT "cveID", "vendorProject", "product", "vulnerabilityName", "dateAdded", "shortDescription", "requiredAction", "dueDate", "knownRansomwareCampaignUse", "notes", "cwes"
FROM "CVE"
WHERE "cwes" LIKE '%CWE-89%'
ORDER BY "dateAdded" DESC;

[2] SQL 실행 중...
    실행 완료

[3] 답변 생성 중...

답변:


최근 순으로 보면, **CWE-89(SQL Injection)** 에 해당하면서 **실제 악용(known ransomware campaign use 또는 명확한 exploitation 가능성 포함)** 이 확인된 취약점들은 아래와 같습니다.

1. **CVE-2026-72898** — Metabase SQL Injection Vulnerability  
   - **등록일:** 2026-08-11  
   - **악용 가능성:** 비인증 원격 공격자가 SQL 주입으로 Metabase 애플리케이션 DB에 접근해 관리자 권한 획득 가능

2. **CVE-2026-60137** — WordPress Core SQL Injection Vulnerability  
   - **등록일:** 2026-07-21  
   - **악용 가능성:** 플러그인/테마가 untrusted input을 전달할 때 SQL Injection 발생,  
     **CVE-2026-63030과 연계 시** 기본 WordPress 설치에서 인증 없이 RCE 가능

3. **CVE-2026-9082** — Drupal Core SQL Injection Vulnerability  
   - **등록일:** 2026-05-22  
   - **악용 가능성:** 특수하게 조작된 요청으로 권한 상승 및 원격 코드 실행 가능

4. **CVE-2026-42208** — BerriAI LiteLLM SQL Injection Vulnerability  
   - **등록일:** 2026-05-08  
   - **악용 가능성:** 프록시 DB 데이터 읽기/변조 가능, 프록시 및 관리 자격 증명까지 영향

5. **CVE-2026-21643** — Fortinet FortiClient EMS SQL Injection Vulnerability  
   - **등록일:** 2026-04-13  
   - **악용 가능성:** 특수 제작된 HTTP 요청으로 비인가 코드/명령 실행 가능

6. **CVE-2024-43468** — Microsoft Configuration Manager SQL Injection Vulnerability  
   - **등록일:** 2026-02-12  
   - **악용 가능성:** 악의적 요청으로 서버에서 명령 실행 가능

7. **CVE-2025-57819** — Sangoma FreePBX Authentication Bypass Vulnerability  
   - **등록일:** 2025-08-29  
   - **악용 가능성:** 충분히 정제되지 않은 사용자 입력으로 관리자 접근 및 DB 변조, RCE 가능  
   - **주의:** CWE-89와 함께 **CWE-288**도 포함

8. **CVE-2025-25257** — Fortinet FortiWeb SQL Injection Vulnerability  
   - **등록일:** 2025-07-18  
   - **악용 가능성:** 비인가 공격자가 HTTP/HTTPS 요청으로 SQL 코드/명령 실행 가능

9. **CVE-2025-25181** — Advantive VeraCore SQL Injection Vulnerability  
   - **등록일:** 2025-03-10  
   - **악용 가능성:** `PmSess1` 파라미터를 통해 원격 SQL 명령 실행 가능

10. **CVE-2020-29574** — CyberoamOS (CROS) SQL Injection Vulnerability  
    - **등록일:** 2025-02-06  
    - **악용 가능성:** WebAdmin에서 비인가 공격자가 원격 SQL 문 실행 가능  
    - **주의:** **Known**(실제 악용 사례 확인)

11. **CVE-2024-9465** — Palo Alto Networks Expedition SQL Injection Vulnerability  
    - **등록일:** 2024-11-14  
    - **악용 가능성:** DB 내용 유출, 비밀번호 해시/디바이스 설정/API 키 노출 가능

12. **CVE-2024-9379** — Ivanti Cloud Services Appliance (CSA) SQL Injection Vulnerability  
    - **등록일:** 2024-10-09  
    - **악용 가능성:** 관리자 인증 후 SQL 명령 실행 가능

13. **CVE-2024-29824** — Ivanti Endpoint Manager (EPM) SQL Injection Vulnerability  
    - **등록일:** 2024-10-02  
    - **악용 가능성:** 동일 네트워크 내부의 비인가 공격자가 코드 실행 가능

14. **CVE-2024-6670** — Progress WhatsUp Gold SQL Injection Vulnerability  
    - **등록일:** 2024-09-16  
    - **악용 가능성:** 단일 사용자 구성에서 암호화된 비밀번호 탈취 가능  
    - **주의:** **Known**(실제 악용 사례 확인)

15. **CVE-2023-48788** — Fortinet FortiClient EMS SQL Injection Vulnerability  
    - **등록일:** 2024-03-25  
    - **악용 가능성:** 비인가 공격자가 SYSTEM 권한으로 명령 실행 가능  
    - **주의:** **Known**(실제 악용 사례 확인)

16. **CVE-2023-46748** — F5 BIG-IP Configuration Utility SQL Injection Vulnerability  
    - **등록일:** 2023-10-31  
    - **악용 가능성:** 인증된 공격자가 시스템 명령 실행 가능

17. **CVE-2021-44026** — Roundcube Webmail SQL Injection Vulnerability  
    - **등록일:** 2023-06-22  
    - **악용 가능성:** search / search_params를 통한 SQL Injection

18. **CVE-2023-34362** — Progress MOVEit Transfer SQL Injection Vulnerability  
    - **등록일:** 2023-06-02  
    - **악용 가능성:** DB 비인가 접근 가능  
    - **주의:** **Known**(대규모 실제 악용으로 매우 유명)

원하시면 제가 다음 중 하나로 다시 정리해드릴 수 있습니다:
- **“Known” 값이 있는 것만 추려서** 보여드리기
- **실제 악용 우선순위 순**으로 다시 정렬하기
- **최근 10개만** 간단히 요약하기

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] CSV 파일 준비 및 데이터 확인 완료
- [ ] CSV 데이터 탐색 및 통계 분석 완료
- [ ] Supabase 연결 완료
- [ ] CSV 데이터 업로드 완료
- [ ] 직접 작성한 SQL 쿼리 테스트 완료 (최소 3개)
- [ ] Text2SQL 함수 구현 및 프롬프트 수정 완료
- [ ] 자연어 질문으로 SQL 생성 테스트 완료
- [ ] 완전한 Text2SQL 시스템 (답변 생성) 테스트 완료
- [ ] 최소 5개 이상의 다양한 질문으로 테스트 완료

---

## 추가 개선 아이디어

1. **프롬프트 개선**: SQL 생성 정확도 향상을 위한 예시 추가
2. **에러 핸들링**: SQL 오류 발생 시 재시도 로직 구현
3. **쿼리 검증**: 생성된 SQL이 안전한지 검증하는 로직 추가
4. **결과 포맷팅**: 테이블 형태로 결과 출력
5. **쿼리 히스토리**: 실행한 쿼리와 결과를 저장하여 재사용
6. **고급 SQL**: 서브쿼리, CTE, 윈도우 함수 활용